# Anomaly & fraud detection: methods and why we chose them

DocSense flags four kinds of problems in approved invoices: **exact/near duplicates**, **image duplicates**, **unusual amounts** and **price creep**. This notebook builds a synthetic spend history with *known* injected anomalies, compares candidate methods on precision/recall, and records the reasoning behind the methods used in `backend/apps/analytics/detectors.py`.

Everything here is deterministic (fixed seeds) and uses the production detector code.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'backend').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'backend'))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import date, timedelta
from apps.analytics import detectors as D
rng = np.random.default_rng(42)
print('backend:', ROOT / 'backend')

backend: C:\Users\Hp\Desktop\AI video editor\docsense\backend


## 1. A synthetic spend history with ground-truth anomalies

12 vendors with very different typical invoice sizes (Rs 400 corner shop up to Rs 90,000 wholesaler), 20-40 invoices each, log-normal totals. We inject three amount anomalies per vendor type (x4-x9 spikes, tiny invoices, abnormal tax ratio) and keep the labels.

In [2]:
rows, labels, vendors = [], {}, {}
next_id = 1
for v in range(1, 13):
    scale = float(np.exp(rng.uniform(np.log(400), np.log(90000))))
    n = int(rng.integers(20, 41)); vendors[v] = scale
    for i in range(n):
        total = float(scale * np.exp(rng.normal(0, 0.18)))
        sub = total / 1.18
        rows.append(D.InvoiceRow(next_id, v, f'V{v}-{i}', date(2024,1,1)+timedelta(days=int(i*(300/n))),
                                 total, sub, total - sub, int(rng.integers(1, 7))))
        labels[next_id] = None; next_id += 1
# inject anomalies
inj = {}
for r in rng.choice(rows, 18, replace=False):
    kind = rng.choice(['spike', 'tiny', 'tax'])
    if kind == 'spike':  r.total = r.total * float(rng.uniform(4, 9)); r.subtotal = r.total/1.18; r.sales_tax = r.total - r.subtotal
    if kind == 'tiny':   r.total = r.total / float(rng.uniform(6, 12)); r.subtotal = r.total/1.18; r.sales_tax = r.total - r.subtotal
    if kind == 'tax':    r.sales_tax = 0.0; r.subtotal = r.total; r.n_items = 35
    inj[r.document_id] = kind
df = pd.DataFrame([r.__dict__ for r in rows])
print(len(df), 'invoices,', len(inj), 'injected anomalies:', pd.Series(inj).value_counts().to_dict())

346 invoices, 18 injected anomalies: {'tiny': 9, 'spike': 6, 'tax': 3}


## 2. EDA: why a single global threshold cannot work

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.log10(df.total), bins=40, color='#6366f1'); ax[0].set_title('log10(total): one fat, multi-modal blob')
df.assign(v=df.vendor_id).boxplot(column='total', by='v', ax=ax[1], showfliers=True); ax[1].set_yscale('log')
ax[1].set_title('per vendor (log scale): different centres and spreads'); plt.suptitle('')
plt.tight_layout(); plt.savefig('/dev/null') if False else plt.show()
print('between-vendor spread of medians: x%.0f' % (df.groupby('vendor_id').total.median().max() / df.groupby('vendor_id').total.median().min()))

between-vendor spread of medians: x74


C:\Users\Hp\AppData\Local\Temp\ipykernel_3560\2866540961.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig('/dev/null') if False else plt.show()


## 3. Comparing amount-outlier methods

* **Global z-score** on totals (the naive baseline)
* **Per-vendor classical z-score** (mean/std)
* **Per-vendor robust z-score** (median/MAD on log totals) - production method #1
* **IsolationForest** on vendor-relative size, tax ratio and item count - production method #2
* **Robust + IsolationForest merged** - what DocSense actually runs

In [4]:
def score(flagged):
    flagged = set(flagged); truth = set(inj)
    tp = len(flagged & truth); p = tp/len(flagged) if flagged else 0; r = tp/len(truth)
    return dict(precision=round(p,3), recall=round(r,3), f1=round(2*p*r/(p+r),3) if p+r else 0, flagged=len(flagged))
res = {}
z = (df.total - df.total.mean())/df.total.std()
res['global z > 3'] = score(df.document_id[z.abs() > 3])
g = df.groupby('vendor_id').total
zc = (df.total - g.transform('mean'))/g.transform('std')
res['per-vendor z > 3'] = score(df.document_id[zc.abs() > 3])
rb = D.robust_outliers(rows); res['per-vendor robust z > 3.5'] = score(f.document_id for f in rb)
iso = D.isolation_outliers(rows, contamination=0.03); res['isolation forest'] = score(f.document_id for f in iso)
merged = D.merge_outliers(rb, iso); res['robust + isolation (production)'] = score(f.document_id for f in merged)
table = pd.DataFrame(res).T; table

,precision,recall,f1,flagged
global z > 3,0.667,0.111,0.190,3.0
per-vendor z > 3,1.000,0.500,0.667,9.0
per-vendor robust z > 3.5,1.000,0.833,0.909,15.0
isolation forest,1.000,0.611,0.759,11.0
robust + isolation (production),1.000,1.000,1.000,18.0


**Reading the table.** A global threshold is dominated by the biggest vendors, so it misses spikes at small vendors and flags ordinary big-vendor invoices. Per-vendor *classical* z-scores fix scale but are themselves distorted by the outliers they are trying to find (the spike inflates the standard deviation: *masking*). Median/MAD does not have that problem. IsolationForest catches what totals alone cannot (zero tax on a normal total, 35 line items) and works for vendors with little history; the union gives the best recall at acceptable precision.

**Caveats.** The injected anomalies are large (x4 to x12), so this is an easy benchmark and the perfect union score should not be read as real-world accuracy. Production also applies a *materiality guard* (an amount outlier must differ from the vendor median by at least 1.8x): it was added after seeding the demo workspace, where short vendor series with natural quantity variation produced statistically significant but business-irrelevant flags. Smaller deviations are deliberately not flagged.

In [5]:
by_kind = {k: [] for k in ('spike','tiny','tax')}
caught = {f.document_id for f in merged}
for did, k in inj.items(): by_kind[k].append(did in caught)
print('recall by anomaly type (merged):', {k: f'{sum(v)}/{len(v)}' for k, v in by_kind.items()})
robust_ids = {f.document_id for f in rb}
print('tax-ratio anomalies found by robust z alone:', sum(1 for d,k in inj.items() if k=='tax' and d in robust_ids), '(expected ~0: totals look normal)')

recall by anomaly type (merged): {'spike': '6/6', 'tiny': '9/9', 'tax': '3/3'}
tax-ratio anomalies found by robust z alone: 0 (expected ~0: totals look normal)


## 4. Image duplicates: choosing the perceptual-hash threshold

We render distinct synthetic invoices, create *re-photographed* copies (JPEG re-encode, slight rescale, brightness shift) and compare 64-bit pHash Hamming distances of true duplicates vs different documents.

In [6]:
import imagehash
from PIL import Image, ImageEnhance
import io
val = sorted((ROOT/'ml'/'data'/'synth'/'none'/'images'/'val').glob('*.jpg'))[:40]
if not val: print('synthetic dataset not built; run python -m docsense_ml.synth.build')
def copy_of(im):
    im = im.resize((int(im.width*0.9), int(im.height*0.9)))
    im = ImageEnhance.Brightness(im).enhance(0.85)
    b = io.BytesIO(); im.save(b, 'JPEG', quality=45); return Image.open(io.BytesIO(b.getvalue()))
ims = [Image.open(p).convert('RGB') for p in val]
h = [str(imagehash.phash(i)) for i in ims]; hc = [str(imagehash.phash(copy_of(i))) for i in ims]
same = [D.hamming(a, b) for a, b in zip(h, hc)]
diff = [D.hamming(h[i], h[j]) for i in range(len(h)) for j in range(i+1, len(h))]
print('true duplicates : max distance', max(same), 'mean', round(np.mean(same),1))
print('different docs  : min distance', min(diff), '5th pct', int(np.percentile(diff,5)), 'mean', round(np.mean(diff),1))
plt.hist(diff, bins=30, alpha=.6, label='different documents'); plt.hist(same, bins=10, alpha=.8, label='re-photographed copies')
plt.axvline(6, color='k', ls='--', label='threshold used (6 bits)'); plt.legend(); plt.xlabel('pHash Hamming distance'); plt.show()
print('with threshold 6: duplicate recall', np.mean(np.array(same) <= 6), '| false-match rate', np.mean(np.array(diff) <= 6))

true duplicates : max distance 4 mean 1.2
different docs  : min distance 2 5th pct 12 mean 27.4
with threshold 6: duplicate recall 1.0 | false-match rate 0.014102564102564103


C:\Users\Hp\AppData\Local\Temp\ipykernel_3560\1512291293.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axvline(6, color='k', ls='--', label='threshold used (6 bits)'); plt.legend(); plt.xlabel('pHash Hamming distance'); plt.show()


## 5. Price creep: sensitivity of the threshold

Series of 5 monthly prices for one item with +/-2% noise; the last price is raised by a known percentage. We sweep the detector's `min_increase`.

In [7]:
def creep_hit(true_pct, thr, trials=200):
    hit = 0
    for t in range(trials):
        base = 500.0; ps = [base*(1+rng.normal(0,0.02)) for _ in range(4)] + [base*(1+true_pct)*(1+rng.normal(0,0.02))]
        its = [D.ItemRow(i+1, 1, 'oil', date(2024,1,1)+timedelta(days=30*i), p) for i, p in enumerate(ps)]
        hit += bool(D.price_creep(its, min_increase=thr))
    return hit/trials
grid = pd.DataFrame({f'min_increase={thr:.0%}': {f'true +{p:.0%}': creep_hit(p, thr) for p in (0.0, 0.05, 0.10, 0.15, 0.30)} for thr in (0.05, 0.08, 0.10, 0.15)})
grid

,min_increase=5%,min_increase=8%,min_increase=10%,min_increase=15%
true +0%,0.02,0.00,0.000,0.00
true +5%,0.56,0.05,0.010,0.00
true +10%,0.98,0.76,0.505,0.02
true +15%,1.00,0.99,0.970,0.52
true +30%,1.00,1.00,1.000,1.00


The 5% rows show why 10% is the default: at +0% the detector must stay quiet (false alarms from 2% noise appear below ~8%), while real +15% and +30% increases are caught essentially always.

## Decisions

| Problem | Method in production | Why |
|---|---|---|
| Exact duplicate | same file hash / same vendor + invoice number | deterministic, zero false positives |
| Near duplicate | same vendor + amount, dates within 3 days | catches re-keyed copies |
| Image duplicate | pHash Hamming <= 6 bits | all re-photographed copies caught; ~1.4% of pairs of *different* synthetic invoices also fall within 6 bits (similar templates), so this is the weakest flag and has `low` severity |
| Unusual amount | per-vendor robust z (median/MAD, log) **+** IsolationForest, merged | section 3: robust to masking; forest covers tax/size combos |
| Price creep | latest unit price >= 10% above the earlier median (>= 3 observations) | section 5 |
